In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 

warnings.filterwarnings('ignore')

# 1. Load the dataset
data = pd.read_csv('insurance.csv')

# 2. Data Cleaning & Handling Duplicates
df_cleaned = data.copy()
df_cleaned.drop_duplicates(inplace=True)

# 3. Encoding Categorical Variables
df_cleaned['sex'] = df_cleaned['sex'].map({'male': 0, 'female': 1})
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes': 1, 'no': 0})

# Rename columns for clarity
df_cleaned.rename(columns={'sex': 'is_female', 'smoker': 'is_smoke'}, inplace=True)

# One-hot encode the 'region' column
df_cleaned = pd.get_dummies(df_cleaned, columns=['region'], drop_first=True)

# 4. Feature Engineering Additions
# Interaction Feature: High risk smoker & high BMI combination (BMI > 30 is considered obese)
df_cleaned['smoker_high_bmi'] = ((df_cleaned['is_smoke'] == 1) & (df_cleaned['bmi'] > 30)).astype(int)

# Family flag: Whether the person has children or not
df_cleaned['has_children'] = (df_cleaned['children'] > 0).astype(int)

# Age category flag: Seniors above 50
df_cleaned['is_senior'] = (df_cleaned['age'] > 50).astype(int)

# 5. Scaling Numerical Features (Age, BMI, Children)
from sklearn.preprocessing import StandardScaler
cols = ['age', 'bmi', 'children']
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])

# Convert boolean/encoded columns to standard integers
for col in ['is_female', 'is_smoke', 'smoker_high_bmi', 'has_children', 'is_senior', 
            'region_northwest', 'region_southeast', 'region_southwest']:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype(int)

# 6. Define Features (X) and Target (y)
X = df_cleaned.drop('charges', axis=1)
y = df_cleaned['charges']

# 7. Train-Test Split (80% Training, 20% Testing)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 8. Train the Linear Regression Model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

# 9. Test and Predict
y_pred = model.predict(X_test)

# Optional: Print out model performance metrics to include in your project report
from sklearn.metrics import mean_squared_error, r2_score
print("R-squared Score:", r2_score(y_test, y_pred))
print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test, y_pred)))

R-squared Score: 0.9067386284491594
Root Mean Squared Error (RMSE): 4139.7273537344245
